In [12]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import requests
import json
import os
import gradio as gr
import finnhub

In [13]:
load_dotenv(override=True)
openai = OpenAI()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

In [9]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_API_KEY")
pushover_url="https://api.pushover.net/1/messages.json"

In [10]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [38]:
from tinydb import TinyDB, Query
mn_db = TinyDB('4_reConnect_wk1_lab4_market_news.json')
meta_db = TinyDB('4_reConnect_wk1_lab4_metadata.json')

In [118]:
import time

def refresh_latest_market_news():
    latest_unix_timestamp = int(time.time())
    stale_threshold = latest_unix_timestamp - 4 * 60 * 60  # Subtract 4 hours in seconds
    MetaQuery = Query()
    metadata = ''
    metadata = meta_db.search(MetaQuery.type == 'mn_meta')
    print(metadata)
    if(metadata):
        if(metadata[0]["lastNewsUpdate"] > stale_threshold):
            print('Not time to update yet. Next update in 4 hours')
            return  {"latest_news": mn_db.all()}
    

    print('News are stale. updating news now')
    latest_news = finnhub_client.general_news('general',min_id=0)
    print(latest_news)
    mn_db.truncate()
    mn_db.insert_multiple(latest_news)
    meta_db.upsert({'type': 'mn_meta', 'lastNewsUpdate': int(time.time())}, MetaQuery.type == 'mn_meta')
    return {"latest_news": latest_news}


refresh_latest_market_news()
#isNewsStale()

[{'type': 'mn_meta', 'lastNewsUpdate': 1771796668}]
Not time to update yet. Next update in 4 hours


{'latest_news': [{'category': 'top news',
   'datetime': 1771779600,
   'headline': 'Gold sheds its safe-haven status. Is it just another momentum play now?',
   'id': 7578138,
   'image': 'https://static2.finnhub.io/file/publicdatany/finnhubimage/market_watch_logo.png',
   'related': '',
   'source': 'MarketWatch',
   'summary': 'Gold has been on a spectacular, record-breaking bull run for much of the past three years — but some of the shine may be coming off the yellow metal, judging by its moves over the past week.',
   'url': 'https://www.marketwatch.com/story/gold-sheds-its-safe-haven-status-is-it-just-another-momentum-play-now-c60af908'},
  {'category': 'top news',
   'datetime': 1771776702,
   'headline': "Here are the 5 big things we're watching in the stock market this week",
   'id': 7578142,
   'image': 'https://image.cnbcfm.com/api/v1/image/108217942-1761669005127-gettyimages-2243373278-AFP_82AM6Y2.jpeg?v=1761669143&w=1920&h=1080',
   'related': '',
   'source': 'CNBC',
   

In [41]:
push("Hey!")

Push: Hey!


In [42]:
# the tools

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}


In [43]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [44]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [48]:
refresh_latest_market_news_json = {
    "name": "refresh_latest_market_news",
    "description": "Always use this tool to get the market news, if any news in our store are more than 4 hours old. If all news are less than 4 hours old, then we serve them from our local store",
    "parameters": {
        "type": "object",
        "properties": {
            },
        },
        "required": [],
        "additionalProperties": False
}

In [66]:
tools = [{"type":"function","function": record_user_details_json}
,{"type":"function","function":record_unknown_question_json}
,{"type":"function","function":refresh_latest_market_news_json}]

In [46]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append([{"role":"tool","content":json.dumps(result), "tool_call_id": tool_call.id}])
    return results

In [101]:
def format_news_array(news_array):
  """
  Formats an array of news objects into a specific text format.

  Args:
    news_array: A list of dictionaries, where each dictionary represents a news article
               and has keys like 'headline', 'summary', and 'url'.

  Returns:
    A string containing the formatted news articles.  Returns an empty string if
    the input array is empty.
  """

  if not news_array:
    return ""

  formatted_text = ""
  for news_item in news_array:
    formatted_text += f"Headline: {news_item['headline']}\n"
    formatted_text += f"Summary: {news_item['summary']}\n"
    formatted_text += f"Url: {news_item['url']}\n"
    formatted_text += "\n"  # Add a blank line between articles

  return formatted_text

ChatCompletion(id='chatcmpl-DCCagkz66kIYFkeqgg8dz71JuWsr8', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_vXzWEyuhNshFLrBo7KdT4lEI', function=Function(arguments='{}', name='refresh_latest_market_news'), type='function')]))], created=1771800706, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_373a14eb6f', usage=CompletionUsage(completion_tokens=12, prompt_tokens=525, total_tokens=537, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
making tool call
Tool called: refresh_latest_market_news
[{'type': 'mn_meta', 'lastNewsUpdate': 1771796668}]
Not time to update

In [114]:
system_prompt = f"""You are acting as a financial news expert. You are answering questions on latest market news in the last 4 hours. \
    You should maintain a professional tone. You should sound like Jim Cramer. \
    Always ask the user for his name and email address, using the tool record_user_details \
    Before greeting the user, refresh the local news store by calling refresh_latest_market_news tool \
    refresh_latest_market_news tool will return an array of objects in the following format: \
        {{'category': 'top news', 'datetime': 1771772401, 'headline': 'Peanut butter pay raises could cost companies their top performers, according to experts: ''It''s such a shortsighted strategy''', 'id': 7578137, 'image': 'https://image.cnbcfm.com/api/v1/image/108266417-1771350651350-gettyimages-816233382-peabu.jpeg?v=1771350664&w=1920&h=1080', 'related': '', 'source': 'CNBC', 'summary': 'In an effort to cut costs, some companies are implementing ''peanut butter'' pay increases. According to experts, it could motivate top performers to leave.', 'url': 'https://www.cnbc.com/2026/02/22/peanut-butter-pay-raises-could-cost-companies-their-top-performers-according-to-experts-its-such-a-shortsighted-strategy.html'}} 
    Greet the user using  with a statement that Jim Crammer would make using the summary and headline properties from all news about market conditions.\
    Use the news returned by refresh_latest_market_news tool to answer user questions for users"""

In [119]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user","content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools = tools)
        print(response)
        finish_reason = response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            print(f'making tool call')
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            print(f"message: {message}")
            print(f"results: {results}")
            # Flatten the list of lists
            flattened_results = [item for sublist in results for item in sublist]
            messages.append(message)
            messages.extend(flattened_results)
        else:
            done = True
    return response.choices[0].message.content

In [122]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(id='chatcmpl-DCQj3wg9hK0d5obvbpkRlEPZXHhKB', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_x2Yz83nkSJS2kMG9mkLFPfce', function=Function(arguments='{}', name='refresh_latest_market_news'), type='function')]))], created=1771855041, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_373a14eb6f', usage=CompletionUsage(completion_tokens=12, prompt_tokens=543, total_tokens=555, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
making tool call
Tool called: refresh_latest_market_news
[{'type': 'mn_meta', 'lastNewsUpdate': 1771811579}]
News are stale. up